# Part 1 - human groundedness evaluation
Both team members scored every final-system (hybrid) answer on the golden set **independently**
(1 = not supported by the cited evidence, 5 = fully supported). This notebook only aggregates `part1/human_eval.csv`.
Runs on CPU in seconds; no model is loaded.

In [1]:
# Run information (required)
import subprocess, sys, random, numpy as np, pandas as pd, os
SEED = 266; random.seed(SEED); np.random.seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit or "no git checkout", "python": sys.version.split()[0], "gpu": "none (CPU only)", "seed": SEED})
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

{'commit': '6a820b6', 'python': '3.13.7', 'gpu': 'none (CPU only)', 'seed': 266}


In [2]:
he = pd.read_csv("part1/human_eval.csv")
gold = pd.read_csv("part1/golden_set.csv", dtype=str)[["gid", "category", "gold_status"]]
res = pd.read_csv("part1/golden_results_hybrid.csv", dtype=str)[["gid", "correct"]]
he = he.merge(gold, on="gid").merge(res, on="gid")
a, b = he["rater_A"].astype(float), he["rater_B"].astype(float)
he["mean"] = (a + b) / 2
print(f"n = {len(he)} answers")
print(f"Groundedness (1-5): rater A {a.mean():.2f}, rater B {b.mean():.2f}, overall {he['mean'].mean():.2f}")

n = 25 answers
Groundedness (1-5): rater A 3.08, rater B 2.48, overall 2.78


In [3]:
# Inter-rater agreement
def quadratic_kappa(x, y, k=5):
    x, y = np.asarray(x, int) - 1, np.asarray(y, int) - 1
    O = np.zeros((k, k))
    for i, j in zip(x, y):
        O[i, j] += 1
    W = np.array([[(i - j) ** 2 / (k - 1) ** 2 for j in range(k)] for i in range(k)])
    E = np.outer(O.sum(1), O.sum(0)) / O.sum()
    return 1 - (W * O).sum() / (W * E).sum()

print(f"Exact agreement: {(a == b).mean():.2f}")
print(f"Within 1 point: {((a - b).abs() <= 1).mean():.2f}")
print(f"Pearson r: {np.corrcoef(a, b)[0, 1]:.2f}")
print(f"Quadratic-weighted Cohen's kappa: {quadratic_kappa(a, b):.2f}")

Exact agreement: 0.24
Within 1 point: 0.56
Pearson r: -0.10
Quadratic-weighted Cohen's kappa: -0.08


In [4]:
# Groundedness by question category and by automatic correctness
print(he.groupby("category")[["rater_A", "rater_B", "mean"]].mean().round(2))
print()
print(he.groupby("correct")[["rater_A", "rater_B", "mean"]].agg(["mean", "count"]).round(2))

              rater_A  rater_B  mean
category                            
conflict         5.00     2.00  3.50
fact             2.60     2.60  2.60
figure           3.50     2.50  3.00
multi_turn       2.40     2.20  2.30
two_source       3.00     2.67  2.83
unanswerable     2.33     3.00  2.67

        rater_A       rater_B        mean      
           mean count    mean count  mean count
correct                                        
False      3.38     8    2.50     8  2.94     8
True       2.94    17    2.47    17  2.71    17


In [5]:
# Largest disagreements (useful to discuss in the report)
he["diff"] = (a - b).abs()
print(he.sort_values("diff", ascending=False)[["gid", "category", "pred_answer", "rater_A", "rater_B"]].head(6).to_string(index=False))

gid     category                                                                                                                                                                                                                                                  pred_answer  rater_A  rater_B
G22     conflict                                                                                                                                                                                                                                                          4.7        5        1
G24 unanswerable                                                                             U.S. federal spending fell as a share of GDP in 2009 because of the economic crisis and the need to counteract the decrease in private demand through increased public spending.        1        4
G21     conflict                                                                                                                        